# Tanager Mangrove Mapping - 03b Multispectral-Equivalent Transferability

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | July 2026 |

---

**Scope (Scenario B — journal publication only):** Apply the XGBoost Tuned model trained on Sangatta Scenario B (`02b`) to 4 transfer sites (Gujarat, El Salvador, Belize, Australia) without retraining. Evaluate against GMW v3 to obtain Kappa B per site. Combine with Kappa A from Scenario A (NB04) to produce the final Objective 3 comparison table across all 5 sites.

**Design principle:** Both scenarios use identical model config (XGBoost Tuned, same hyperparameters). Both use the same coastal candidate mask logic and adaptive threshold per scene. Only the feature stack differs: Scenario A uses 6 features (NDMI, MVI, MNDWI, SAVI, EMI, REIP); Scenario B uses 4 features (NDMI, MVI, MNDWI, SAVI). Delta Kappa (B - A) per site therefore isolates the contribution of REIP and EMI to zero-shot transferability.

## 0. Environment Setup

In [ ]:
# !pip install scikit-learn geopandas rasterio matplotlib joblib xgboost


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
import json
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

# ============================================================
# Project root and paths
# ============================================================
# Google Colab (Google Drive mounted)
ROOT           = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')

# Local (uncomment if running locally)
# ROOT = Path('..').resolve()
DATA_PROC      = ROOT / 'data' / 'processed'
DATA_PROC_S2EQ = ROOT / 'data' / 'processed_s2eq'
DATA_GMW       = ROOT / 'data' / 'gmw_v3'
OUT_MODELS     = ROOT / 'outputs' / 'models'
OUT_RESULTS    = ROOT / 'outputs' / 'results'
OUT_FIGURES    = ROOT / 'outputs' / 'figures'

sys.path.insert(0, str(ROOT))

# ============================================================
# Reload src modules during development
# ============================================================
import src.classification  as _cls
import src.evaluation      as _eval
import src.transferability as _tr
import src.spatial_viz     as _sviz
importlib.reload(_cls)
importlib.reload(_eval)
importlib.reload(_tr)
importlib.reload(_sviz)

from src.classification  import load_model, predict_extent
from src.evaluation      import (
    rasterize_gmw,
    evaluate_against_gmw,
    plot_confusion_matrix,
    plot_agreement_map,
)
from src.transferability import TRANSFER_SITES, SITE_LABELS
from src.spatial_viz     import reproject_raster_to_4326, format_map_axes

# ============================================================
# Training site + transfer sites
# ============================================================
TRAIN_SITE     = 'sangatta'
TRAIN_SCENE_ID = '20250302_030003_92_4001'

print(f'ROOT           : {ROOT}')
print(f'Training site  : {TRAIN_SITE} ({TRAIN_SCENE_ID})')
print(f'Transfer sites : {list(TRANSFER_SITES.keys())}')
print(f'Feature stack B: NDMI, MVI, MNDWI, SAVI (4 features)')


## 1. Load Scenario B Trained Model

XGBoost Tuned model trained on Sangatta Scenario B pseudo-labels in `02b_classification.ipynb`. Applied as-is to all transfer sites (zero-shot, no retraining).

In [ ]:
# ============================================================
# Load Scenario B XGBoost Tuned model
# ============================================================
model_path = OUT_MODELS / f'xgb_tuned_s2eq_{TRAIN_SITE}_{TRAIN_SCENE_ID}.joblib'
xgb_b      = load_model(str(model_path))

print(f'Model loaded   : {model_path.name}')
print(f'Model type     : {type(xgb_b).__name__}')


## 2. Transfer Loop -- 4 Sites

Per-site pipeline (no retraining):
1. Load Scenario B indices from `data/processed_s2eq/{site_key}_s2eq_indices.tif` (precomputed in `01b`)
2. Load candidate mask from `data/processed_s2eq/candidate_s2eq_{site}_{scene_id}.tif` (from `01b`)
3. Predict extent with Scenario B model (constrained to candidate mask)
4. Save extent GeoTIFF
5. Rasterize GMW v3 and evaluate → Kappa B per site

In [ ]:
# ============================================================
# Transfer loop -- 4 sites
# ============================================================
import time

transfer_results_b = []

for site, scene_id in TRANSFER_SITES.items():
    print(f'\n{"="*60}')
    print(f'  Transfer site  : {SITE_LABELS[site]}  ({scene_id})')
    print(f'{"="*60}')
    t0 = time.time()

    site_key = f'{site}_{scene_id}'

    # 1. Load Scenario B indices
    s2eq_path = DATA_PROC_S2EQ / f'{site_key}_s2eq_indices.tif'
    with rasterio.open(s2eq_path) as src:
        arr    = src.read().astype(np.float32)
        data_b = {'transform': src.transform, 'crs': src.crs}

    indices_b = {
        'NDMI'  : arr[0],
        'MVI'   : arr[1],
        'MNDWI' : arr[2],
        'SAVI'  : arr[3],
    }

    # 2. Load candidate mask
    cand_path = DATA_PROC_S2EQ / f'candidate_s2eq_{site}_{scene_id}.tif'
    with rasterio.open(cand_path) as src:
        candidate_mask = src.read(1).astype(bool)

    # 3. Predict extent (4 features, no extra_features)
    h, w     = indices_b['NDMI'].shape
    extent_b = predict_extent(
        xgb_b, indices_b,
        original_shape=(h, w),
        candidate_mask=candidate_mask,
        extra_features=None,
    )

    # 4. Save extent
    extent_path = DATA_PROC_S2EQ / f'extent_mangrove_s2eq_{site}_{scene_id}.tif'
    with rasterio.open(
        extent_path, 'w',
        driver='GTiff', height=h, width=w,
        count=1, dtype='int8',
        crs=data_b['crs'], transform=data_b['transform'],
        compress='lzw', nodata=-1,
    ) as dst:
        dst.write(extent_b, 1)

    n_mang  = int((extent_b == 1).sum())
    area_ha = n_mang * 0.09
    print(f'  Extent saved   : {extent_path.name}')
    print(f'  Mangrove area  : {area_ha:,.1f} ha  ({n_mang:,} px)')

    result = {
        'site'          : site,
        'site_label'    : SITE_LABELS[site],
        'scene_id'      : scene_id,
        'n_mangrove_px' : n_mang,
        'area_ha'       : area_ha,
    }

    # 5. Evaluate vs GMW v3
    gmw_path = DATA_GMW / f'gmw_{site}_{scene_id}.geojson'
    if gmw_path.exists():
        gmw_raster = rasterize_gmw(
            str(gmw_path),
            reference_shape=(h, w),
            transform=data_b['transform'],
            crs=data_b['crs'],
        )
        metrics = evaluate_against_gmw(
            extent_b, gmw_raster,
            eval_mask=candidate_mask,
            model_name=f'XGBoost Tuned S2eq ({SITE_LABELS[site]})',
        )
        metrics['metrics_table'].to_csv(
            OUT_RESULTS / f'gmw_eval_s2eq_{site}_{scene_id}.csv', index=False
        )
        result.update({
            'kappa'     : metrics['kappa'],
            'precision' : metrics['precision'],
            'recall'    : metrics['recall'],
            'f1'        : metrics['f1_mangrove'],
            'iou'       : metrics['IoU'],
        })
    else:
        print(f'  GMW v3 not found: {gmw_path.name} -- skipping eval')
        result.update({'kappa': None, 'precision': None,
                       'recall': None, 'f1': None, 'iou': None})

    print(f'  Duration       : {time.time()-t0:.1f} s')
    transfer_results_b.append(result)


## 3. Scenario B Summary Across All 5 Sites

Combines Sangatta (training, from `02b`) and 4 transfer sites (from Section 2 above).

In [ ]:
# ============================================================
# Load Sangatta Scenario B metrics (from 02b)
# ============================================================
sangatta_b_path = OUT_RESULTS / f'gmw_eval_s2eq_{TRAIN_SITE}_{TRAIN_SCENE_ID}.csv'
df_sangatta_b   = pd.read_csv(sangatta_b_path)

# Normalize column names to match transfer schema
col_map = {'Kappa': 'kappa', 'Precision': 'precision', 'Recall': 'recall',
           'F1_mangrove': 'f1', 'IoU': 'iou'}
df_sangatta_b = df_sangatta_b.rename(columns=col_map)

sangatta_row = {
    'site'       : TRAIN_SITE,
    'site_label' : f'Sangatta (train)',
    'scene_id'   : TRAIN_SCENE_ID,
    'kappa'      : df_sangatta_b['kappa'].values[0],
    'precision'  : df_sangatta_b['precision'].values[0],
    'recall'     : df_sangatta_b['recall'].values[0],
    'f1'         : df_sangatta_b['f1'].values[0],
    'iou'        : df_sangatta_b['iou'].values[0],
}

# Combine with transfer results
all_rows_b = [sangatta_row] + [
    {k: v for k, v in r.items() if k in sangatta_row}
    for r in transfer_results_b
]
df_all_b = pd.DataFrame(all_rows_b)

# Round for display
for c in ['kappa', 'precision', 'recall', 'f1', 'iou']:
    if c in df_all_b.columns:
        df_all_b[c] = df_all_b[c].round(4)

print('\n  Scenario B -- all 5 sites (XGBoost Tuned, 4 features):')
print(df_all_b[['site_label', 'kappa', 'precision', 'recall', 'f1', 'iou']].to_string(index=False))

df_all_b.to_csv(OUT_RESULTS / 'transferability_summary_s2eq.csv', index=False)
print('\n  Saved : transferability_summary_s2eq.csv')


## 4. Final Comparison: Kappa A vs Kappa B (All 5 Sites)

The Objective 3 result. Compares XGBoost Tuned Scenario A (6 features) vs Scenario B (4 features) across the training site and 4 transfer sites. Positive delta (B - A) indicates that REIP and EMI reduce accuracy at that site; negative delta indicates they help.

**Data sources:**
- Scenario A Sangatta: `model_comparison_sangatta_*.csv` (NB02), XGBoost Tuned row
- Scenario A transfer sites: `transferability_summary_xgboost.csv` (NB04)
- Scenario B all sites: `transferability_summary_s2eq.csv` (Section 3 above)

In [ ]:
# ============================================================
# Build final comparison table Kappa A vs Kappa B (5 sites)
# ============================================================

# --- Scenario A: Sangatta from model_comparison (XGBoost Tuned row) ---
comp_a_path = OUT_RESULTS / f'model_comparison_{TRAIN_SITE}_{TRAIN_SCENE_ID}.csv'
df_comp_a   = pd.read_csv(comp_a_path)
row_xgb_a_sangatta = df_comp_a[df_comp_a['Model'] == 'XGBoost Tuned'].iloc[0]

sangatta_a = {
    'site_label' : 'Sangatta (train)',
    'kappa_A'    : float(row_xgb_a_sangatta['Kappa']),
    'kappa_B'    : df_sangatta_b['kappa'].values[0],
}

# --- Scenario A: transfer sites from transferability_summary_xgboost.csv ---
transfer_a_path = OUT_RESULTS / 'transferability_summary_xgboost.csv'
df_transfer_a   = pd.read_csv(transfer_a_path)

# --- Scenario B: transfer sites from Section 2 above ---
df_transfer_b = pd.DataFrame(transfer_results_b)

# --- Merge per site ---
comparison_rows = [sangatta_a]

for site, scene_id in TRANSFER_SITES.items():
    site_label = SITE_LABELS[site]

    # Scenario A row -- match by site label from NB04 output
    row_a = df_transfer_a[df_transfer_a['site'] == site_label]
    kappa_a = float(row_a['kappa'].values[0]) if not row_a.empty else None

    # Scenario B row -- match by site key
    row_b = df_transfer_b[df_transfer_b['site'] == site]
    kappa_b = float(row_b['kappa'].values[0]) if not row_b.empty else None

    comparison_rows.append({
        'site_label' : site_label,
        'kappa_A'    : kappa_a,
        'kappa_B'    : kappa_b,
    })

df_final = pd.DataFrame(comparison_rows)
df_final['delta_B_A'] = df_final['kappa_B'] - df_final['kappa_A']

# Round for display
for c in ['kappa_A', 'kappa_B', 'delta_B_A']:
    df_final[c] = df_final[c].round(4)

print('\n  Kappa A vs Kappa B -- All 5 sites (XGBoost Tuned, same hyperparameters):')
print(df_final.to_string(index=False))

df_final.to_csv(OUT_RESULTS / 'kappa_A_vs_B_all_sites.csv', index=False)
print('\n  Saved : kappa_A_vs_B_all_sites.csv')

# Summary: how many sites where B > A, B < A, tie
n_b_higher = int((df_final['delta_B_A'] > 0).sum())
n_a_higher = int((df_final['delta_B_A'] < 0).sum())
n_tie      = int((df_final['delta_B_A'] == 0).sum())
mean_delta = df_final['delta_B_A'].mean()

print(f'\n  Sites where Scenario B > Scenario A : {n_b_higher} / {len(df_final)}')
print(f'  Sites where Scenario A > Scenario B : {n_a_higher} / {len(df_final)}')
print(f'  Ties                                : {n_tie} / {len(df_final)}')
print(f'  Mean delta (B - A)                  : {mean_delta:+.4f}')


## 5. Visualization -- Multi-Site Extent Maps (Scenario B)

Same layout as `04_transferability.ipynb` Section 4 for direct visual comparison against Scenario A.

In [ ]:
# ============================================================
# Multi-site extent maps -- Scenario B
# ============================================================
sites_to_plot = list(TRANSFER_SITES.items())
n_sites       = len(sites_to_plot)
fig, axes     = plt.subplots(1, n_sites, figsize=(5 * n_sites, 6))

for ax, (site, scene_id) in zip(axes, sites_to_plot):
    extent_path = DATA_PROC_S2EQ / f'extent_mangrove_s2eq_{site}_{scene_id}.tif'
    if not extent_path.exists():
        ax.text(0.5, 0.5, f'{site}\nnot found',
                ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        continue
    with rasterio.open(extent_path) as src:
        n_px = int((src.read(1) == 1).sum())
        _img, _ext = reproject_raster_to_4326(src)
    ax.imshow(_img == 1, extent=_ext, origin='upper', cmap='Greens')
    ax.set_aspect('equal')
    ax.set_title(f'{SITE_LABELS[site]}\n({n_px:,} px)')
    format_map_axes(ax, fontsize=8)

plt.suptitle('Mangrove Extent -- Transfer Sites (Scenario B: 4 features, XGBoost Tuned, trained on Sangatta)', y=1.02)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'transferability_extent_maps_s2eq.png', dpi=150, bbox_inches='tight')
plt.show()
